## Integrated Gradients - Pathway Enrichment

- Author: Zhao

I have alrealy implemented Integrated gradients for Deepsynegry and found all top 20 feature, which are most relevant for the synergy score.

My idea is that we find the similarities of those top 20 features: What kinds of biological/genomic similarities -> Biological Relevance

I used:

- Analyse Biological Relevance with **Pathway Enrichment**
    - g:Profiler 

as methods.


### Preparation

This script maps “feature indices” back to their human-readable names and categories.

In [ ]:
import gzip
import pickle
import numpy as np

# 1. Load configuration and data
with gzip.open("test0val1normtanh_norm.p.gz", "rb") as f:
    X_tr, X_val, X_train, X_test, \
    y_tr, y_val, y_train, y_test, \
    index_names, f_feature_origin, f_feature_group = pickle.load(f)

# 2. Count how many features in each module
groups, counts = np.unique(f_feature_group, return_counts=True)
print("Feature groups and counts:", dict(zip(groups, counts)))
# → e.g. {'ECFP_6': 2618, 'genomic': 3984, 'phys-chem': 230, 'toxicophore': 2014}

# 3. For each module, print its index range and total feature count
groups_array = np.array(f_feature_group)
for grp in groups:
    idxs = np.where(groups_array == grp)[0]
    start, end, cnt = idxs.min(), idxs.max(), idxs.size
    print(f"{grp:12s}: indices {start:4d}-{end:4d} (total {cnt} features)")

# 4. Extract and display genomic feature names
mask = groups_array == "genomic"
genomic_indices = np.where(mask)[0]
print("\nGenomic features index range:",
      genomic_indices.min(), "to", genomic_indices.max())

gene_names = [f_feature_origin[i] for i in genomic_indices]
#print("First 10 genomic feature names:", gene_names[:10])

# 5. Map a list of top feature indices back to their names and groups
top_indices = [
    8006, 2017, 6943, 1950, 5049,
    5930, 2135, 1695, 5681, 4538,
    4545, 5395, 4623, 4185, 8087,
    1981, 2226, 5570, 5864, 2248
]
print("\nTop feature mappings:")
for idx in top_indices:
    name = f_feature_origin[idx]
    grp  = f_feature_group[idx]
    print(f"  Feature {idx:4d} -> {name} (group: {grp})")


### Outputs:

In [ ]:
Feature groups and counts: {np.str_('ECFP_6'): np.int64(2618), np.str_('genomic'): np.int64(3984), np.str_('phys-chem'): np.int64(230), np.str_('toxicophore'): np.int64(2014)}
ECFP_6      : indices    0-3739 (total 2618 features)
genomic     : indices 4862-8845 (total 3984 features)
phys-chem   : indices 1309-3854 (total 230 features)
toxicophore : indices 1424-4861 (total 2014 features)

Genomic features index range: 4862 to 8845

Top feature mappings:
  Feature 8006 -> genomic_11918 (group: genomic)
  Feature 2017 -> toxicophore_3971 (group: toxicophore)
  Feature 6943 -> genomic_10855 (group: genomic)
  Feature 1950 -> toxicophore_3904 (group: toxicophore)
  Feature 5049 -> genomic_8961 (group: genomic)
  Feature 5930 -> genomic_9842 (group: genomic)
  Feature 2135 -> toxicophore_4089 (group: toxicophore)
  Feature 1695 -> toxicophore_3640 (group: toxicophore)
  Feature 5681 -> genomic_9593 (group: genomic)
  Feature 4538 -> toxicophore_8448 (group: toxicophore)
  Feature 4545 -> toxicophore_8455 (group: toxicophore)
  Feature 5395 -> genomic_9307 (group: genomic)
  Feature 4623 -> toxicophore_8533 (group: toxicophore)
  Feature 4185 -> toxicophore_8086 (group: toxicophore)
  Feature 8087 -> genomic_11999 (group: genomic)
  Feature 1981 -> toxicophore_3935 (group: toxicophore)
  Feature 2226 -> toxicophore_4180 (group: toxicophore)
  Feature 5570 -> genomic_9482 (group: genomic)
  Feature 5864 -> genomic_9776 (group: genomic)
  Feature 2248 -> toxicophore_4202 (group: toxicophore)

There are 9 genomic features and 11 toxicophore features in the top 20 features 

### Pathway Enrichment for genomic features 


**What is pathway?:** A pathway in biology is simply a series of molecular “steps” or interactions inside a cell that lead to a specific outcome.

In [ ]:
import gzip
import pickle
import numpy as np
import pandas as pd
from gprofiler import GProfiler

# ———— Configuration ———— #
PKL_FILE    = "test0val1normtanh_norm.p.gz"
# Top feature indices obtained from IG/PI/PDP analysis
TOP_INDICES = [8006, 2017, 6943, 1950, 5049, 2135, 1695, 4538, 4545, 4623, 4185, 1981, 2226, 2248]
ORGANISM    = "hsapiens"
OUT_TSV     = "gprofiler_results.tsv"
# —————————————— #

# 1. Unpack to get feature origin names and groups
with gzip.open(PKL_FILE, "rb") as f:
    *_, f_feature_origin, f_feature_group = pickle.load(f)

# 2. Identify the index range for genomic features
groups = np.array(f_feature_group)
genomic_mask = groups == "genomic"
genomic_indices = np.where(genomic_mask)[0]

# 3. Filter TOP_INDICES for genomic features and extract gene IDs
top_genes = []
for idx in TOP_INDICES:
    grp = f_feature_group[idx]
    if grp != "genomic":
        print(f"Feature {idx} belongs to '{grp}', skipping")
        continue
    orig = f_feature_origin[idx]
    # orig looks like "genomic_12345"; split and keep the ID part
    gene_id = orig.split("_", 1)[1]
    top_genes.append(gene_id)

print("✔ Final gene list for enrichment:", top_genes)

if not top_genes:
    raise ValueError("No genomic features found. Please check TOP_INDICES and f_feature_group.")

# 4. Call g:Profiler for enrichment analysis
gp = GProfiler(return_dataframe=True)
res = gp.profile(
    organism=ORGANISM,
    query=top_genes,
    sources=['GO:BP', 'GO:MF', 'GO:CC', 'KEGG', 'REAC']
)

# 5. Filter significant results (FDR < 0.05) and save to file
sig = res[res['p_value'] < 0.05].sort_values('p_value')
sig.to_csv(OUT_TSV, sep="\t", index=False)
print(f"Enrichment results saved to {OUT_TSV}")

# 6. Print all enrichment terms
if sig.empty:
    print("No significant enrichment (FDR<0.05). Consider relaxing the threshold or increasing the number of top features.")
else:
    display_cols = ['source', 'native', 'name', 'p_value', 'term_size', 'query_size', 'intersection_size']
    print("\n=== All enrichment results ===")
    print(sig[display_cols].to_string(index=False))


### Results

Outcome pathways: they are most relevant for our model.

1. Glycosaminoglycan degradation
2. heparanase activity

In [ ]:
source	native	name	p_value	significant	description	term_size	query_size	intersection_size	effective_domain_size	precision	recall	query	parents
KEGG	KEGG:00531	Glycosaminoglycan degradation	0.03647409878261118	True	Glycosaminoglycan degradation	19	1	1	8484	1.0	0.05263157894736842	query_1	['KEGG:00000']
GO:MF	GO:0030305	heparanase activity	0.04951475539710648	True	"""Catalysis of the cleavage of heparan sulfate; can degrade both heparan sulfate and heparin glycosaminoglycan chains."" [PMID:10916150]"	2	1	1	20196	1.0	0.5	query_1	['GO:0004553']


### Week 9

**try to find out,if p values correct?**

->  multiple testing 

In [2]:
import pandas as pd
from statsmodels.stats.multitest import multipletests

# Read the enrichment results from g:Profiler output
res = pd.read_csv(
    "/Users/guoguo/Desktop/Studium/6.Semester/SWP/gprofiler_results.tsv",
    sep="\t"
)

# Extract the original p-values
pvals = res['p_value'].values

# Apply Benjamini–Hochberg FDR correction
reject, qvals, _, _ = multipletests(
    pvals,
    alpha=0.05,
    method='fdr_bh'
)
# Store corrected q-values and significance flags
res['q_value']    = qvals        # Adjusted FDR q-values
res['signif_fdr'] = reject       # True if q_value < 0.05

# Display key columns: term ID, original p-value, corrected q-value, and FDR significance
print(res[['native', 'p_value', 'q_value', 'signif_fdr']].to_string(index=False))


    native  p_value  q_value  signif_fdr
KEGG:00531 0.036474 0.049515        True
GO:0030305 0.049515 0.049515        True


<span style="color:blue">Conclusion: P-Values are significant</span>。

###  Enrichment for toxicophores


- have to generate a SMARTS list-- <span style="color:red">toxicophore definition：can't do this </span>。
    - Our Preprocessor class never actually creates the toxicophore bits (chemical structures)
    - Without that original SMARTS list (Expression of chemical structures) in our preprocessing code, there is no way to recover the chemical meaning of toxicophores 
    
    
- need to use over representation -> <span style="color:red">see below</span>。


In [ ]:
import gzip
import pickle
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# 1. Load feature origin names and groups
with gzip.open("test0val1normtanh_norm.p.gz", "rb") as f:
    *_, f_feature_origin, f_feature_group = pickle.load(f)
groups = np.array(f_feature_group)

# 2. Prepare background and query sets
#    Indices of all toxicophore bits in the background
bg_idxs = np.where(groups == "toxicophore")[0]
bg_bits = [f_feature_origin[i] for i in bg_idxs]
#    Query set: select toxicophore bits from TOP_INDICES
TOP_INDICES = [8006, 2017, 6943, 1950, 5049, 2135, 1695, 4538, 4545, 4623, 4185, 1981, 2226, 2248]
top_bits = [f_feature_origin[i] for i in TOP_INDICES if f_feature_group[i] == "toxicophore"]

# 3. Count occurrences of each bit in background and top sets
from collections import Counter
bg_count = Counter(bg_bits)
top_count = Counter(top_bits)

results = []
N_bg = len(bg_bits)   # Total number of background bits (≈2014)
K_top = len(top_bits) # Number of bits in the query set

for bit, x in top_count.items():
    M = bg_count[bit]  # Count of this bit in the background (usually 1)
    # Build contingency table: [[in_top, not_in_top], [in_bg, not_in_bg]]
    table = [[x, K_top - x],
             [M - x, N_bg - M - (K_top - x)]]
    p = fisher_exact(table, alternative="greater")[1]
    results.append((bit, x, M, p))

# 4. Multiple testing correction
df = pd.DataFrame(results, columns=["bit", "top_count", "bg_count", "p_value"])
reject, qvals, _, _ = multipletests(df["p_value"], method="fdr_bh")
df["q_value"] = qvals
df["signif_fdr"] = reject

# 5. Output the results
df = df.sort_values("p_value")
print(df.to_string(index=False))

### Result of over representation:

In [ ]:
             bit  top_count  bg_count  p_value  q_value  signif_fdr
toxicophore_3971          1         1 0.005462 0.005462        True
toxicophore_3904          1         1 0.005462 0.005462        True
toxicophore_4089          1         1 0.005462 0.005462        True
toxicophore_3640          1         1 0.005462 0.005462        True
toxicophore_8448          1         1 0.005462 0.005462        True
toxicophore_8455          1         1 0.005462 0.005462        True
toxicophore_8533          1         1 0.005462 0.005462        True
toxicophore_8086          1         1 0.005462 0.005462        True
toxicophore_3935          1         1 0.005462 0.005462        True
toxicophore_4180          1         1 0.005462 0.005462        True
toxicophore_4202          1         1 0.005462 0.005462        True

- Each of these 11 toxicophore bits appears exactly once in my selected “top” list -> top 20 features list from IG.
- Each bit also exists only once in the full background of 2,014 possible toxicophore bits.

**Summary:**

Because each bit is extremely rare in the background (only one copy), the moment it shows up in my top 11, it achieves “significant” over-representation. In other words, these bits aren’t “more important” than each other—they’re simply rare and happened to be selected.

-> This result seems to be unnecessary for me.

**or maybe not correct....**